# Hands-on: The 4 Core Capabilities of an Agent
## Learning AI Agents with LangGraph & LangChain

This notebook walks through the 4 core capabilities of Agentic AI step by step, using a **real LLM and LangGraph**.
The default LLM is the free cloud API **NVIDIA build + `deepseek-ai/deepseek-v4.1-flash`** (no local server needed),
and **OpenRouter** (`openrouter/free`) is the comparison target you can switch to with a single line, `LLM_PROVIDER`, in `.env` (see §0-B).

| # | Capability | Key concept | Exercises |
|---|------|----------|---------|
| 1 | **Tool Use** | The LLM calls external tools | @tool decorator, bind_tools(), automatic loop |
| 2 | **Memory** | Retaining and using conversation history | Message history, LangGraph MemorySaver |
| 3 | **Planning** | Breaking complex problems into steps | Structured output (Pydantic), automatic plan execution |
| 4 | **Reasoning** | Logical step-by-step reasoning | Chain of Thought, LangGraph ReAct Agent |
| 5 | **Integration** | Combining the 4 capabilities | create_agent + MemorySaver |

> 📦 **Environment installation and run commands** are documented separately in [`env_guides/M01_2_core_capabilities_eng.md`](env_guides/M01_2_core_capabilities_eng.md).
> Repeated/shared code has been factored out into the [`agentic_lib/`](agentic_lib) library (tools · memory · planning · react · bootstrap · capabilities).

### Prerequisites
- Completed `M01_1_intro_eng.ipynb` (uv environment set up, `.env` file created)
- Default: `LLM_PROVIDER=nvidia` + `NVIDIA_API_KEY` in `notebooks/.env` (free from build.nvidia.com)
- (Optional) Comparison: `LLM_PROVIDER=openrouter` + `OPENROUTER_API_KEY` (free from openrouter.ai)


---
## 0. Environment Setup


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(''))  # Add notebooks/ to the import path

import utils
utils.reload_env()  # Reload .env (refreshes LLM_PROVIDER, etc.) + print the current provider status

from utils import (
    uv_install, get_llm, test_llm_connection, LLM_PROVIDER,
    NVIDIA_API_KEY, NVIDIA_MODEL,   # For connecting to NVIDIA build (free credits) — §0-B
)

# Repeated/shared code is factored out into the agentic_lib library (used throughout this notebook).
from agentic_lib import bootstrap, tools, memory, planning, react
from agentic_lib.bootstrap import to_text          # Provider-agnostic response normalization (including <think> removal)
from agentic_lib.capabilities import (             # Advanced Planning: structured output schema + executor
    PlanStep, ExecutionPlan, Status, execute_plan,
)

# Install LangGraph / LangChain packages
uv_install([
    'langgraph>=0.2.0',
    'langchain>=0.3.0',
    'langchain-core>=0.3.0',
    'pydantic>=2.0.0',
])

llm = test_llm_connection()
print(f'\nReady: using the {LLM_PROVIDER} provider')


LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


[uv] 설치 완료: ['langgraph>=0.2.0', 'langchain>=0.3.0', 'langchain-core>=0.3.0', 'pydantic>=2.0.0']


LLM 연결 성공 [nvidia]: 1+1은 2가 됩니다.

준비 완료: nvidia 공급자 사용


---
## 0-B. Calling the NVIDIA build Cloud Model Directly

Let's directly call a model on **NVIDIA build** (build.nvidia.com, free credits), this notebook's default LLM.
`utils.get_llm('nvidia')` returns a `ChatNVIDIA`, so it works with the **same interface** (`.invoke()` / `bind_tools()`) as the rest of this notebook's code.

| Item | Value |
|---|---|
| Connection | `utils.get_llm('nvidia')` (= `ChatNVIDIA`) |
| Key | `NVIDIA_API_KEY` in `.env` (free from build.nvidia.com) |
| Model | `NVIDIA_MODEL` in `.env` (default `deepseek-ai/deepseek-v4.1-flash`) |

- **Individual call**: use it right away with `nvidia_llm = utils.get_llm('nvidia')`, as in the cell below.
- **Whole notebook on OpenRouter**: change to `LLM_PROVIDER=openrouter` in `.env` and re-run from the §0 cell; `llm` itself then becomes the OpenRouter model (no code changes needed).
- Without a key, only **key-issuing guidance** is printed, with no error. For more detailed examples, see [`M02_0_free_llm_api_eng.ipynb`](M02_0_free_llm_api_eng.ipynb).

In [2]:
# === Demo: calling the NVIDIA build cloud model directly ===
# Install the NVIDIA connector (passes quickly if already installed)
uv_install(['langchain-nvidia-ai-endpoints'])

from langchain_core.messages import HumanMessage, SystemMessage

nvidia_llm = None
if not NVIDIA_API_KEY:
    print('NVIDIA_API_KEY not set — get one for free at https://build.nvidia.com and add it to .env.')
else:
    try:
        nvidia_llm = get_llm('nvidia')                 # Returns ChatNVIDIA(model=NVIDIA_MODEL)
        demo = nvidia_llm.invoke([
            SystemMessage(content='Answer concisely in English.'),
            HumanMessage(content='Explain LangGraph in one sentence.'),
        ])
        print(f'[NVIDIA build · {NVIDIA_MODEL}]')
        print(' ', to_text(demo.content))              # Provider-agnostic response normalization
    except Exception as e:
        print(f'[NVIDIA call failed] {type(e).__name__}: {e}')
        print('  → Check that the model name is offered on build.nvidia.com and that your key/credits are valid.')

# Note: this notebook's Tool Use and agent cells can use nvidia_llm in the same way
#      (when the NVIDIA model supports tool calling). e.g. llm_with_tools = nvidia_llm.bind_tools(tool_list)
#      Since the default is LLM_PROVIDER=nvidia, the llm from §0 is the same NVIDIA model.

[uv] 설치 완료: ['langchain-nvidia-ai-endpoints']


[NVIDIA build · meta/llama-3.1-8b-instruct]
  LangGraph는 언어 모델링을 위한 그래프 기반의 프레임워크로, 자연어 처리 및 언어 모델링을 위한 다양한 기능을 제공합니다.


---
## 1. Tool Use — How an LLM Uses External Tools

Tool Use is the ability of an LLM to go beyond plain text generation and **directly call external functions (tools)**.

```
User question
    ↓
LLM analysis → Which tool? With which arguments?
    ↓
Tool (function) execution
    ↓
Pass the execution result to the LLM
    ↓
Generate the final natural-language answer
```

**Key APIs**
- `@tool` decorator: turns a regular Python function into a LangChain tool
- `llm.bind_tools(tools)`: passes the tool list to the LLM (schemas generated automatically)
- `ToolMessage`: passes the tool execution result back to the LLM as a message


### 1-1. Defining Tools (the `@tool` decorator)


In [3]:
# Shared tools come from agentic_lib.tools (calculator / get_current_time / search_web).
# (Instead of redefining tools in each notebook, they are managed in one place in the library — duplicate definitions removed)
tool_list = tools.BASIC_TOOLS               # [calculator, get_current_time, search_web]
tool_map = {t.name: t for t in tool_list}   # Name → tool mapping (used to execute tools in the cells below)

print('Registered tools:')
for t in tool_list:
    print(f'  - {t.name}: {t.description}')


등록된 도구 목록:
  - calculator: 수학 수식을 계산합니다. 예: '2 ** 10', 'sqrt(144)', '(3+4)*5'.

    Python 수식 문법을 사용하며 sqrt, sin, log, pi 같은 math 함수/상수를 쓸 수 있습니다.
  - get_current_time: 현재 날짜와 시간을 'YYYY-MM-DD HH:MM:SS' 형식으로 반환합니다.

    인자가 없는 도구를 허용하지 않는 일부 공급자를 위해 사용하지 않는 dummy 인자를 둡니다.
  - search_web: 웹을 검색합니다. 기본은 DuckDuckGo 실제 검색이며, 오프라인/실패 시 시뮬레이션으로 폴백합니다.

    - DuckDuckGo(ddgs)는 API 키가 필요 없습니다.
    - 재현 가능한 결정론적 실행을 원하면 환경변수 `WEB_SEARCH_MODE=sim` 로 시뮬레이션을 강제하세요.
    - ddgs 미설치·네트워크 차단·레이트리밋 시에는 자동으로 시뮬레이션 결과로 폴백합니다.


### 1-2. Binding Tools to the LLM — Single Tool Call Flow


In [4]:
from langchain_core.messages import HumanMessage, ToolMessage

# 1) Bind the tool list to the LLM
llm_with_tools = llm.bind_tools(tool_list)

question = 'What is 2 to the power of 15?'
print(f'[Question] {question}\n')

# 2) The LLM decides which tool to call with which arguments
response = llm_with_tools.invoke([HumanMessage(content=question)])

print(f'[LLM response] number of tool_calls: {len(response.tool_calls)}')
if response.tool_calls:
    tc = response.tool_calls[0]
    print(f"  → Selected tool  : {tc['name']}")
    print(f"  → Tool arguments : {tc['args']}")

    # 3) Execute the tool
    tool_result = tool_map[tc['name']].invoke(tc['args'])
    print(f'  → Tool result    : {tool_result}')

    # 4) Pass the tool result to the LLM → final natural-language answer
    final_messages = [
        HumanMessage(content=question),
        response,
        ToolMessage(content=str(tool_result), tool_call_id=tc['id'])
    ]
    final_resp = llm_with_tools.invoke(final_messages)
    print(f'\n[Final answer] {to_text(final_resp.content)}')  # Normalize list/think with to_text


[질문] 2의 15제곱이 얼마야?



[LLM 응답] tool_calls 수: 1개
  → 선택 도구 : calculator
  → 도구 인자 : {'expression': '2 ** 15'}
  → 도구 결과 : 2 ** 15 = 32768



[최종 답변] 2의 15제곱은 32,768입니다.


### 1-3. Automatic Tool Call Loop — Handling Multiple Tools in Sequence


In [5]:
from langchain_core.messages import AIMessage  # Record parallel tool_calls split into single-call turns

def run_tool_agent(question: str, max_steps: int = 6) -> str:
    """Take a question, repeatedly auto-select and execute tools, and return the final answer.

    Even if a single response contains multiple tool_calls, each call is recorded in the history as a
    pair of 'a separate AIMessage holding a single tool_call + its corresponding ToolMessage'. This way
    it works without errors even on models that 'support only one tool_call at a time', such as
    llama-3.1 on NVIDIA build (`This model only supports single tool-calls at once!`).
    """
    print(f'[Question] {question}')
    print('-' * 55)

    messages = [HumanMessage(content=question)]

    for step in range(1, max_steps + 1):
        response = llm_with_tools.invoke(messages)

        # No tool call → final answer
        if not response.tool_calls:
            messages.append(response)
            answer = to_text(response.content)  # Provider-agnostic normalization
            print(f'[Final answer] {answer}')
            return answer

        # Handle tool calls — record each tool_call as a single-call turn (compatible with single-call-only models)
        for tc in response.tool_calls:
            print(f"[Step {step}] {tc['name']}({tc['args']})")
            messages.append(AIMessage(content='', tool_calls=[tc]))
            result = tool_map[tc['name']].invoke(tc['args'])
            print(f'         → {result}')
            messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))

    return 'Maximum steps exceeded'


print('=' * 60)
print('Test 1: Using multiple tools together')
print('=' * 60)
run_tool_agent('What time is it now? Also, calculate 3 to the power of 7.')

print()
print('=' * 60)
print('Test 2: Using the search tool')
print('=' * 60)
run_tool_agent('Search for information about LangGraph.')

테스트 1: 여러 도구 동시 사용
[질문] 지금 몇 시야? 그리고 3의 7제곱도 계산해줘.
-------------------------------------------------------


[Step 1] get_current_time({'dummy': ''})
         → 2026-07-25 15:50:39
[Step 1] calculator({'expression': '3 ** 7'})
         → 3 ** 7 = 2187


[최종 답변] The current time is 2026-07-25 15:50:39 and 3 to the power of 7 is 2187.

테스트 2: 검색 도구 사용
[질문] LangGraph 에 대해 검색해줘.
-------------------------------------------------------


[Step 1] search_web({'query': 'LangGraph'})


         → 'LangGraph' 웹 검색 결과 상위 5건:
1. Laographer
   Folklore is the body of expressive culture shared by a particular group of people, culture or subculture. This includes oral traditions such as tales, legends, 
   https://en.wikipedia.org/wiki/Laographer
2. LangGraph
   LangGraph is an MIT-licensed open-source low-level orchestration framework developed by the LangChain team for building resilient, stateful, multi-actor AI agen
   https://grokipedia.com/page/LangGraph
3. LangGraph: Agent Orchestration Framework for Reliable AI Agents
   Build controllable agents with LangGraph, our low-level agent orchestration framework
   https://www.langchain.com/langgraph
4. GitHub - langchain-ai/langgraph: Build resilient agents. · GitHub
   May 6, 2026 - Trusted by companies shaping the future of agents – including Klarna, Replit, Elastic, and more – LangGraph is a low-level orchestration framework
   https://github.com/langchain-ai/langgraph
5. What is LangGraph? | IBM
   2 weeks ago - Lang

[Step 2] search_web({'query': 'LangGraph'})


         → 'LangGraph' 웹 검색 결과 상위 5건:
1. Laographer
   Folklore is the body of expressive culture shared by a particular group of people, culture or subculture. This includes oral traditions such as tales, legends, 
   https://en.wikipedia.org/wiki/Laographer
2. LangGraph
   LangGraph is an MIT-licensed open-source low-level orchestration framework developed by the LangChain team for building resilient, stateful, multi-actor AI agen
   https://grokipedia.com/page/LangGraph
3. LangGraph: Agent Orchestration Framework for Reliable AI Agents
   Build controllable agents with LangGraph, our low-level agent orchestration framework
   https://www.langchain.com/langgraph
4. GitHub - langchain-ai/langgraph: Build resilient agents. · GitHub
   May 6, 2026 - Trusted by companies shaping the future of agents – including Klarna, Replit, Elastic, and more – LangGraph is a low-level orchestration framework
   https://github.com/langchain-ai/langgraph
5. What is LangGraph? | IBM
   2 weeks ago - Lang

[최종 답변] The search results for "LangGraph" are as follows:

1. Laographer
   Folklore is the body of expressive culture shared by a particular group of people, culture or subculture. This includes oral traditions such as tales, legends, 
   https://en.wikipedia.org/wiki/Laographer

2. LangGraph
   LangGraph is an MIT-licensed open-source low-level orchestration framework developed by the LangChain team for building resilient, stateful, multi-actor AI agen
   https://grokipedia.com/page/LangGraph

3. LangGraph: Agent Orchestration Framework for Reliable AI Agents
   Build controllable agents with LangGraph, our low-level agent orchestration framework
   https://www.langchain.com/langgraph

4. GitHub - langchain-ai/langgraph: Build resilient agents. · GitHub
   May 6, 2026 - Trusted by companies shaping the future of agents – including Klarna, Replit, Elastic, and more – LangGraph is a low-level orchestration framework
   https://github.com/langchain-ai/langgraph

5. What is LangGraph? |

'The search results for "LangGraph" are as follows:\n\n1. Laographer\n   Folklore is the body of expressive culture shared by a particular group of people, culture or subculture. This includes oral traditions such as tales, legends, \n   https://en.wikipedia.org/wiki/Laographer\n\n2. LangGraph\n   LangGraph is an MIT-licensed open-source low-level orchestration framework developed by the LangChain team for building resilient, stateful, multi-actor AI agen\n   https://grokipedia.com/page/LangGraph\n\n3. LangGraph: Agent Orchestration Framework for Reliable AI Agents\n   Build controllable agents with LangGraph, our low-level agent orchestration framework\n   https://www.langchain.com/langgraph\n\n4. GitHub - langchain-ai/langgraph: Build resilient agents. · GitHub\n   May 6, 2026 - Trusted by companies shaping the future of agents – including Klarna, Replit, Elastic, and more – LangGraph is a low-level orchestration framework\n   https://github.com/langchain-ai/langgraph\n\n5. What is L

---
## 2. Memory — An Agent That Remembers

An AI Agent's memory is divided into two layers.

| Type | Scope | Implementation |
|------|------|----------|
| **Short-term memory** | Current conversation session | Message history (Python list) |
| **Long-term memory** | Persists across sessions | LangGraph `MemorySaver` / external DB |

LangGraph automatically saves the graph's execution state through a **Checkpointer**
and distinguishes users/sessions by `thread_id`.


### 2-1. Short-Term Memory — Message History


In [6]:
from langchain_core.messages import SystemMessage, HumanMessage

print('=' * 60)
print('Short-term memory: keeping conversation context with a message list')
print('=' * 60)

conversation = [
    SystemMessage(content='You are an AI assistant that carefully remembers user information.')
]

def chat(user_input: str) -> str:
    """Append the user input to the conversation list, get the LLM response, and accumulate it."""
    conversation.append(HumanMessage(content=user_input))
    reply = llm.invoke(conversation)
    conversation.append(reply)
    return to_text(reply.content)  # Return a string normalized for output


turns = [
    'Hello! I am Jisoo Park and I live in Busan.',
    'I am a data scientist studying LangGraph.',
    'Where did I say I live and what do I do?',
]

for i, user_msg in enumerate(turns, 1):
    print(f'\n[Turn {i}]')
    print(f'  User: {user_msg}')
    reply = chat(user_msg)
    print(f'  Agent: {reply[:120]}')

print(f'\nTotal accumulated messages: {len(conversation)}')


단기 기억: 메시지 리스트로 대화 맥락 유지

[대화 1]
  사용자: 안녕하세요! 저는 박지수이고 부산에 살아요.


  에이전트: 안녕하세요 박지수 씨! 부산에 살고 계신다는 것을 알게 되었습니다. 부산은 좋은 도시라고 들었는데요. 어떤 활동이나 취미가 있으신가요?

[대화 2]
  사용자: 저는 데이터 사이언티스트로 LangGraph 를 공부하고 있어요.


  에이전트: 데이터 사이언티스트로 LangGraph를 공부하고 계신다는 것을 알게 되었습니다. LangGraph는 자연어 처리와 언어 모델링에 사용되는 그래프 기반의 언어 모델링 기술입니다. LangGraph를 공부하시는 것은 

[대화 3]
  사용자: 제가 어디 살고 무슨 일을 한다고 했었죠?


  에이전트: 박지수 씨라고 하셨고, 부산에 살고 계신다는 것을 기억했습니다. 또한 데이터 사이언티스트로 LangGraph를 공부하고 계신다는 것을 알게 되었습니다.

누적 메시지 수: 7개


### 2-2. Long-Term Memory — LangGraph MemorySaver

`MemorySaver` **persistently stores in memory** the graph's state (all messages) per `thread_id`.
As long as the process is alive, memory is retained even across sessions.
(In production, you can swap it for `SqliteSaver` / `PostgresSaver` for durable storage)


In [7]:
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import MemorySaver

# ── Graph definition ─────────────────────────────────────────
def chatbot_node(state: MessagesState) -> dict:
    """A single node that calls the LLM with the system prompt + accumulated messages."""
    system = SystemMessage(content='You are an AI assistant that remembers user information.')
    response = llm.invoke([system] + state['messages'])
    return {'messages': [response]}

builder = StateGraph(MessagesState)
builder.add_node('chatbot', chatbot_node)
builder.add_edge(START, 'chatbot')

memory_saver = MemorySaver()
app = builder.compile(checkpointer=memory_saver)

# ── Utility function ─────────────────────────────────────────
def chat_with_thread(thread_id: str, message: str) -> str:
    """Hold one conversation turn while keeping state per thread_id."""
    config = {'configurable': {'thread_id': thread_id}}
    result = app.invoke({'messages': [HumanMessage(content=message)]}, config=config)
    return to_text(result['messages'][-1].content)  # Normalized string


print('=' * 60)
print('LangGraph MemorySaver: independent memory per thread_id')
print('=' * 60)

# Thread 'alice': Alice's conversation
print('\n[Thread: alice]')
r = chat_with_thread('alice', "Hi! I'm Alice. I live in New York.")
print(f'  1st: {r[:80]}')

r = chat_with_thread('alice', "I'm studying AI agents.")
print(f'  2nd: {r[:80]}')

r = chat_with_thread('alice', 'Where did I say I live and what am I studying?')
print(f'  3rd: {r[:120]}')

# Thread 'bob': Bob's conversation (completely independent of Alice)
print('\n[Thread: bob — no information about Alice]')
r = chat_with_thread('bob', 'Where did I say I live and what am I studying?')
print(f'  Answer: {r[:100]}')


LangGraph MemorySaver: thread_id 별 독립 기억

[Thread: alice]


  1번째: 안녕하세요 앨리스! 뉴욕에 살고 계신가요? 뉴욕은 정말 멋진 도시입니다. 어떤 활동이나 관심사를 가지고 계신가요?


  2번째: AI 에이전트를 공부하시는군요. AI는 정말 흥미로운 분야입니다. 뉴욕에 살면서 AI를 공부하시는 것은 정말 좋은 기회인 것 같습니다. 어떤 부


  3번째: 앨리스님께서 뉴욕에 살고 AI 에이전트를 공부한다고 말씀하셨습니다.

[Thread: bob — 앨리스 정보 없음]


  답변: 당신이 이전에 말한 정보를 기억하고 있지 않습니다. 이전에 어떤 정보를 말한 적이 없기 때문에 기억할 수 없습니다. 하지만 지금부터는 당신과 대화할 때 기억할 수 있습니다.


### 2-3. Checking Memory State per Thread


In [8]:
from langchain_core.messages import AIMessage

print('=' * 60)
print('Checking saved conversation history')
print('=' * 60)

for thread_id in ['alice', 'bob']:
    config = {'configurable': {'thread_id': thread_id}}
    state = app.get_state(config)
    msgs = state.values.get('messages', [])
    print(f'\n[{thread_id}] number of messages: {len(msgs)}')
    for msg in msgs:
        role = 'User' if isinstance(msg, HumanMessage) else 'Agent'
        print(f'  [{role}] {to_text(msg.content)[:70]}')  # Print after normalizing content


저장된 대화 히스토리 확인

[alice] 메시지 수: 6개
  [사용자] 안녕! 나는 앨리스야. 뉴욕에 살고 있어.
  [에이전트] 안녕하세요 앨리스! 뉴욕에 살고 계신가요? 뉴욕은 정말 멋진 도시입니다. 어떤 활동이나 관심사를 가지고 계신가요?
  [사용자] AI 에이전트를 공부하고 있어.
  [에이전트] AI 에이전트를 공부하시는군요. AI는 정말 흥미로운 분야입니다. 뉴욕에 살면서 AI를 공부하시는 것은 정말 좋은 기회인 것 
  [사용자] 내가 어디 살고 뭘 공부한다고 했지?
  [에이전트] 앨리스님께서 뉴욕에 살고 AI 에이전트를 공부한다고 말씀하셨습니다.

[bob] 메시지 수: 2개
  [사용자] 내가 어디 살고 뭘 공부한다고 했지?
  [에이전트] 당신이 이전에 말한 정보를 기억하고 있지 않습니다. 이전에 어떤 정보를 말한 적이 없기 때문에 기억할 수 없습니다. 하지만 지


---
## 3. Planning — An Agent That Plans

Planning is the ability to **break a complex goal down into executable steps**.

Using LangChain's **Structured Output**,
the LLM returns its plan as **JSON conforming to a Pydantic schema** instead of free text.

```
Goal input
    ↓
LLM (calls the schema as a tool) → Pydantic object
    ↓
Automatic step-by-step execution (tools / LLM)
    ↓
Collect the final results
```


### 3-1. Generating an Execution Plan with Structured Output


In [9]:
# The advanced Planning implementation (PlanStep / ExecutionPlan / Status / execute_plan) is
# factored out into agentic_lib.capabilities (already imported at the top).

# Apply a structured output schema to the LLM — respond with an ExecutionPlan (JSON) instead of free text
# If the ExecutionPlan schema is bound as a 'tool', the LLM fills in the plan as the tool's arguments (JSON) and calls it,
# and PydanticToolsParser validates and converts those arguments into an ExecutionPlan object.
# Note: llm.with_structured_output(ExecutionPlan) is simpler, but in that case ChatNVIDIA sends the NVIDIA-specific
#       guided_json parameter, which the deepseek-v4.1-flash endpoint rejects ([400] unknown field).
#       The tool-calling approach works on both NVIDIA and OpenRouter.
from langchain_core.output_parsers import PydanticToolsParser
planner = llm.bind_tools([ExecutionPlan]) | PydanticToolsParser(tools=[ExecutionPlan], first_tool_only=True)

# ── Grounding in the current time ─────────────────────────────────────────
# An LLM does not know 'today's date' beyond its training cutoff (e.g. around 2023 for llama-3.1).
# So when asked about time-related fields (estimated_duration, etc.), it makes up past dates like 2023.
# Getting the actual current time with the get_current_time tool and injecting it into the prompt makes it
# answer relative to 'now', producing correct values instead of stale 2023 dates.
now = tool_map['get_current_time'].invoke({'dummy': ''})   # e.g. '2026-07-11 11:23:02'

plan_request = (
    f'Current time: {now}\n'
    'Goal: Write an analysis report on AI agent technology trends\n'
    'Available tools: calculator (math calculations), get_current_time (current time), search_web (web search)\n'
    'Write a 5-step plan that uses the appropriate tool at each step.\n'
    'Fill estimated_duration not with a date but with the "expected time required" relative to the current time above (e.g. "about 3 hours", "2 days").'
)

plan: ExecutionPlan = planner.invoke(plan_request)
if plan is None:   # When the model answered only with text without calling the tool (schema)
    raise RuntimeError('The LLM did not produce an ExecutionPlan. Try running the cell again.')

print(f'[Reference current time] {now}')            # The actual current time injected into the prompt
print(f'[Goal] {plan.goal}')
print(f'[Estimated duration] {plan.estimated_duration}')
print(f'\n[Execution plan] — {len(plan.steps)} steps:')
for step in plan.steps:
    dep = f'  (depends on: Step {step.depends_on})' if step.depends_on else ''
    tool_tag = f'  [tool: {step.tool}]' if step.tool else ''
    print(f'  Step {step.step_num}: {step.action}{tool_tag}{dep}')
    print(f'    Expected output: {step.expected_output}')

[기준 현재 시각] 2026-07-25 15:50:53
[목표] AI 에이전트 기술 동향 분석 보고서 작성
[예상 소요 시간] 약 5일

[실행 계획] — 5단계:
  Step 1: 과제 정의 및 목표 설정
    예상 결과: 과제 정의서 및 목표 설정서 작성
  Step 2: 현재 기술 동향 분석
    예상 결과: AI 에이전트 기술 동향 분석 보고서 초안 작성
  Step 3: 웹 검색 및 데이터 수집
    예상 결과: AI 에이전트 기술 동향 분석 보고서 주요 데이터 수집
  Step 4: 보고서 작성 및 편집
    예상 결과: AI 에이전트 기술 동향 분석 보고서 최종본 작성
  Step 5: 보고서 검토 및 최종 제출
    예상 결과: AI 에이전트 기술 동향 분석 보고서 최종 제출


### 3-2. Automatic Plan Execution


In [10]:
# execute_plan is factored out into agentic_lib.capabilities (runs tools/LLM in dependency order).
# Call it with tool_map and llm; it prints step-by-step progress and returns a results dictionary.
print('=' * 60)
print('Automatic plan execution')
print('=' * 60)
results = execute_plan(plan, tool_map, llm)


계획 자동 실행
[목표] AI 에이전트 기술 동향 분석 보고서 작성

[실행중] Step 1: 과제 정의 및 목표 설정


  → 과제 정의 및 목표 설정은 프로젝트를 성공적으로 완료하기 위한 첫 번째 단계입니다. 과제 정의 및 목표 설정을 통해 프로젝트의 목적, 범위, 시간표, 자원, 그리고 성공 기준을 명

[실행중] Step 2: 현재 기술 동향 분석


  → 현재 기술 동향은 빠르게 변화하고 있습니다. 하지만 몇 가지 주요한 기술 동향을 살펴보겠습니다.

1. **인공지능 (AI) 및 머신 러닝 (ML)**: 인공지능과 머신 러닝은 최

[실행중] Step 3: 웹 검색 및 데이터 수집


  → 웹 검색 및 데이터 수집은 웹에서 데이터를 수집하고 분석하는 과정입니다. 이 과정을 통해 웹에서 다양한 데이터를 수집하고 분석할 수 있습니다. 웹 검색 및 데이터 수집의 예를 들어

[실행중] Step 4: 보고서 작성 및 편집


  → 보고서 작성 및 편집은 중요한 업무입니다. 보고서 작성은 정보를 정리하고 분석하여 보고서를 작성하는 과정을 의미하며, 보고서 편집은 작성된 보고서를 검토하고 수정하여 최종 버전을 

[실행중] Step 5: 보고서 검토 및 최종 제출


  → 보고서 검토 및 최종 제출을 위한 단계는 다음과 같습니다.

1. **보고서 검토**: 
 - 보고서의 내용을 확인하여 정확성과 완전성을 확인합니다.
 - 보고서의 구조와 형식이 

[완료] 5/5 단계 실행


---
## 4. Reasoning — An Agent That Reasons

Reasoning is the ability to **analyze given information logically and think step by step**.

| Technique | Description | Effect |
|------|------|------|
| **Chain of Thought (CoT)** | "Think step by step" prompt | Improves accuracy on complex math and logic problems |
| **ReAct** | Loop combining Reasoning + Acting | Solves problems by alternating tool use and reasoning |
| **Self-Reflection** | Re-examines its own answer | Fewer errors, higher quality |


### 4-1. Chain of Thought — Direct Answer vs. Step-by-Step Reasoning


In [11]:
from langchain_core.prompts import ChatPromptTemplate

problem = (
    'The school library has 1,500 books.\n'
    '- Monday: students borrowed 20% of all the books.\n'
    '- Tuesday: another 25% of the remaining books were borrowed.\n'
    '- Wednesday: half of the borrowed books were returned.\n\n'
    'How many books are in the library in total after Wednesday?'
)

# Method A: direct answer (no reasoning)
print('=' * 60)
print('Method A: Direct answer (no reasoning process)')
print('=' * 60)
direct = llm.invoke([HumanMessage(content=problem)])
print(to_text(direct.content)[:300])  # Print after normalizing

# Method B: Chain of Thought
print('\n' + '=' * 60)
print('Method B: Chain of Thought — step-by-step reasoning')
print('=' * 60)

cot_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a precise math problem solver.\n'
     'You must reason step by step in the following format:\n\n'
     '**Step 1:** [First calculation — state the formula and result]\n'
     '**Step 2:** [Second calculation]\n'
     '...\n'
     '**Verification:** [Brief review of the solution]\n'
     '**Final answer:** [Numeric conclusion]'),
    ('human', '{problem}')          # {problem} is a placeholder to be filled in later
])

# ── cot_chain = cot_prompt | llm ──────────────────────────────────────────
# LCEL (LangChain Expression Language) syntax. The '|' (pipe) connects prompt → LLM into a single
# execution chain. '|' feeds the output of the left side into the input of the right side (same idea as a shell pipe).
#   {'problem': ...} ─▶ cot_prompt (fills the template) ─▶ completed messages ─▶ llm (call) ─▶ AIMessage
# In other words, cot_chain is a single composite Runnable that does 'input dict → complete prompt → call LLM → response'.
# (ChatPromptTemplate and llm are both Runnables, so they can be piped together, and the result is again a Runnable)
cot_chain = cot_prompt | llm

# A single .invoke() internally runs two steps automatically: (1) fill the prompt → (2) call the LLM.
# Written out without the pipe, it is equivalent to the two lines below:
#     messages = cot_prompt.invoke({'problem': problem})   # Step 1: complete the prompt
#     cot_result = llm.invoke(messages)                    # Step 2: call the LLM
# Building it as a chain makes it easy to reuse by just changing the problem, or to extend (e.g. | StrOutputParser()).
cot_result = cot_chain.invoke({'problem': problem})
print(to_text(cot_result.content))  # Print after normalizing

방법 A: 직접 답변 (추론 과정 없음)


월요일에 학생들이 20%의 책을 빌렸다면, 20%는 0.2로 계산됩니다. 

1,500 * 0.2 = 300권

월요일에 학생들이 빌린 책의 수는 300권입니다. 

도서관에 남은 책의 수는 1,500 - 300 = 1,200권입니다.

화요일에 남은 책의 25%가 추가로 빌려졌습니다. 

1,200 * 0.25 = 300권

화요일에 추가로 빌린 책의 수는 300권입니다. 

도서관에 남은 책의 수는 1,200 - 300 = 900권입니다.

수요일에 빌린 책의 절반, 즉 300/2 = 150권이 반납되었습니다.

도서관에 남은 

방법 B: Chain of Thought — 단계별 추론


**단계 1:** 월요일에 학생들이 빌린 책의 수를 계산합니다. 
월요일에 학생들이 빌린 책의 수 = 전체 책의 20% = 1,500 * 0.2 = 300권

**단계 2:** 월요일에 학생들이 빌린 책을 제외한 남은 책의 수를 계산합니다.
남은 책의 수 = 전체 책의 수 - 월요일에 학생들이 빌린 책의 수 = 1,500 - 300 = 1,200권

**단계 3:** 화요일에 추가로 빌린 책의 수를 계산합니다.
화요일에 추가로 빌린 책의 수 = 남은 책의 수 * 25% = 1,200 * 0.25 = 300권

**단계 4:** 화요일에 추가로 빌린 책을 포함한 현재 빌린 책의 수를 계산합니다.
현재 빌린 책의 수 = 월요일에 학생들이 빌린 책의 수 + 화요일에 추가로 빌린 책의 수 = 300 + 300 = 600권

**단계 5:** 수요일에 반납된 책의 수를 계산합니다.
수요일에 반납된 책의 수 = 현재 빌린 책의 수 * 50% = 600 * 0.5 = 300권

**단계 6:** 수요일에 반납된 책을 제외한 현재 빌린 책의 수를 계산합니다.
현재 빌린 책의 수 = 현재 빌린 책의 수 - 수요일에 반납된 책의 수 = 600 - 300 = 300권

**단계 7:** 수요일 이후 도서관에 있는 책의 수를 계산합니다.
수요일 이후 도서관에 있는 책의 수 = 전체 책의 수 - 현재 빌린 책의 수 = 1,500 - 300 = 1,200권

**검증:** 계산 과정을 확인하여 오류가 없는지 확인합니다.

**최종 답:** 1,200


### 4-2. The ReAct Pattern — LangGraph's Built-in Agent

`create_agent` implements the **Reasoning + Acting** loop as a LangGraph graph.

```
[Thought] What needs to be done? Which tool is needed?
    ↓
[Action] Tool call
    ↓
[Observation] Check the tool's result
    ↓
(repeat or finish)
    ↓
[Final Answer] Final natural-language answer
```


In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

# ── Agent model ─────────────────────────────────────────────────────────────
# ReAct and unified agents run a 'multi-step tool loop (repeated Thought→Action→Observation)', so
# they need a model that is strong at tool calling. This notebook's default llm, NVIDIA build's
# deepseek-ai/deepseek-v4.1-flash, can call multiple tools in parallel in a single response and handles multi-step
# tool loops reliably, so we use the default llm for the agent as-is, without a separate model.
# Note: the former default model meta/llama-3.1-8b-instruct (service ended 2026-07) allowed only 'one tool_call at a time'
#       and had a weak agent loop, so this cell alone had to be run with a larger model swapped in.
agent_llm = llm
print(f'[Agent model] using the default {LLM_PROVIDER} llm')

react_memory = MemorySaver()

react_agent = create_agent(
    model=agent_llm,               # Default llm with reliable tool calling
    tools=tool_list,
    checkpointer=react_memory,
    system_prompt=(
        'You are an AI agent that solves problems step by step using tools. '
        'Actively use the tools you need and answer in English.'
    ),
)


def run_react_agent(question: str, thread_id: str = 'react-1') -> str:
    """Run the ReAct agent and print the intermediate tool calls, results, and final answer."""
    print(f'[Question] {question}')
    print('-' * 55)

    config = {'configurable': {'thread_id': thread_id}}
    result = react_agent.invoke(
        {'messages': [HumanMessage(content=question)]},
        config=config
    )

    # Show the intermediate process (tool calls)
    from langchain_core.messages import AIMessage, ToolMessage as TM
    for msg in result['messages']:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [ReAct → tool selected] {tc['name']}({tc['args']})")
        elif isinstance(msg, TM):
            print(f'  [ReAct → tool result] {to_text(msg.content)[:80]}')

    final = to_text(result['messages'][-1].content)  # Print after normalizing
    print(f'\n[Final answer] {final}')
    return final


print('=' * 60)
print('ReAct Test 1: Tool reasoning + calculation')
print('=' * 60)
run_react_agent(
    'What time is it now? Calculate the remainder when the minute of that time is divided by 7.',
    thread_id='react-test-1'
)

print('\n' + '=' * 60)
print('ReAct Test 2: Search + summarize')
print('=' * 60)
run_react_agent(
    'Search for AI trends and summarize the key points in 3 lines.',
    thread_id='react-test-2'
)

[에이전트 모델] qwen/qwen3-next-80b-a3b-instruct (기본 llm=llama-3.1-8b-instruct 대신 사용)
ReAct 테스트 1: 도구 추론 + 계산
[질문] 지금 몇 시야? 그 시각의 분(minute)을 7로 나눈 나머지를 계산해줘.
-------------------------------------------------------


  [ReAct → 도구 선택] get_current_time({})
  [ReAct → 도구 결과] 2026-07-25 15:52:02
  [ReAct → 도구 선택] calculator({'expression': '52 % 7'})
  [ReAct → 도구 결과] 52 % 7 = 3

[최종 답변] 현재 시각은 2026년 7월 25일 15시 52분입니다.  
분(minute)인 52를 7로 나눈 나머지는 3입니다.

ReAct 테스트 2: 검색 + 요약
[질문] AI 트렌드에 대해 검색하고 핵심을 3줄로 요약해줘.
-------------------------------------------------------


  [ReAct → 도구 선택] search_web({'query': '최신 AI 트렌드 2024 핵심 동향'})
  [ReAct → 도구 결과] '최신 AI 트렌드 2024 핵심 동향' 웹 검색 결과 상위 5건:
1. 미래를 여는 열쇠 AGI: 관련 산업과 핵심 기업들의 역할
   최신 
  [ReAct → 도구 선택] search_web({'query': '2024년 AI 트렌드 멀티모델 오케스트레이션 자율 에이전트'})
  [ReAct → 도구 결과] '2024년 AI 트렌드 멀티모델 오케스트레이션 자율 에이전트' 웹 검색 결과 상위 5건:
1. AI 에이전트 오케스트레이션 완벽 가이드 202
  [ReAct → 도구 선택] search_web({'query': '2024년 AI 트렌드 신뢰성과 윤리적 AI 실용화'})
  [ReAct → 도구 결과] '2024년 AI 트렌드 신뢰성과 윤리적 AI 실용화' 웹 검색 결과 상위 5건:
1. 2024년 예상되는 AI 기술 트렌드 12가지 - 디지털

[최종 답변] 2024년 AI 트렌드의 핵심은 다음과 같습니다:

1. **멀티모델 오케스트레이션**이 주류로 부상: GPT-5, Claude, Gemini 등 다양한 AI 모델을 협업시켜 복잡한 작업을 자동으로 분배하고 실행하는 아키텍처가 기업과 연구기관에서 실용화되고 있습니다.

2. **자율 에이전트의 확산**: 단순한 응답을 넘어, 목표를 설정하고 도구를 선택하며 환경과 상호작용하는 자율 AI 에이전트가 마케팅, 개발, 고객 서비스 등 다양한 산업에 적용되고 있습니다.

3. **신뢰성과 윤리적 AI 강화**: 투명성, 편향 제거, 데이터 프라이버시를 고려한 윤리적 AI 개발이 기술적 진보와 병행되며, 규제와 표준화가 전 세계적으로 가속화되고 있습니다.


'2024년 AI 트렌드의 핵심은 다음과 같습니다:\n\n1. **멀티모델 오케스트레이션**이 주류로 부상: GPT-5, Claude, Gemini 등 다양한 AI 모델을 협업시켜 복잡한 작업을 자동으로 분배하고 실행하는 아키텍처가 기업과 연구기관에서 실용화되고 있습니다.\n\n2. **자율 에이전트의 확산**: 단순한 응답을 넘어, 목표를 설정하고 도구를 선택하며 환경과 상호작용하는 자율 AI 에이전트가 마케팅, 개발, 고객 서비스 등 다양한 산업에 적용되고 있습니다.\n\n3. **신뢰성과 윤리적 AI 강화**: 투명성, 편향 제거, 데이터 프라이버시를 고려한 윤리적 AI 개발이 기술적 진보와 병행되며, 규제와 표준화가 전 세계적으로 가속화되고 있습니다.'

---
## 5. Unified Agent — A Complete Agent Combining the 4 Capabilities

We combine the 4 capabilities learned so far into a single agent.

| Capability | Implementation |
|------|----------|
| Tool Use | `bind_tools` / `create_agent` |
| Memory | `MemorySaver` + `thread_id` |
| Planning | The LLM's inherent planning ability |
| Reasoning | ReAct loop (Thought → Action → Observation) |


In [13]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

# ── Agent model ─────────────────────────────────────────────────────────────
# The unified agent is also a multi-step tool loop, like the ReAct agent in §4-2. The default llm (NVIDIA build's
# deepseek-ai/deepseek-v4.1-flash) handles parallel tool calls and memory recall reliably, so
# we use it as-is.
agent_llm = llm
print(f'[Agent model] using the default {LLM_PROVIDER} llm')

unified_memory = MemorySaver()

SYSTEM_PROMPT = (
    'You are an AI agent equipped with 4 core capabilities.\n\n'
    'Capabilities:\n'
    '- Tool Use  : actively use the calculator, get_current_time, and search_web tools\n'
    '- Memory    : remember the conversation context and respond consistently\n'
    '- Planning  : break complex requests down into steps and handle them\n'
    '- Reasoning : reason logically and provide accurate answers\n\n'
    'Rules:\n'
    '- Always answer in English.\n'
    '- Use tools without hesitation when needed.\n'
    '- When multiple steps are needed, proceed in order.'
)

unified_agent = create_agent(
    model=agent_llm,               # Default llm with reliable tool calling
    tools=tool_list,
    checkpointer=unified_memory,
    system_prompt=SYSTEM_PROMPT,
)


def ask_agent(message: str, thread_id: str = 'unified') -> str:
    """Send a message to the unified agent, print the tool calls and results, then return the answer."""
    config = {'configurable': {'thread_id': thread_id}}
    result = unified_agent.invoke(
        {'messages': [HumanMessage(content=message)]},
        config=config
    )
    from langchain_core.messages import AIMessage, ToolMessage as TM
    for msg in result['messages']:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [Tool] {tc['name']}({tc['args']})")
        elif isinstance(msg, TM):
            print(f'  [Result] {to_text(msg.content)[:80]}')
    return to_text(result['messages'][-1].content)  # Return after normalizing


# ── Composite task 1: Tool Use + Reasoning ───────────────────────────────
print('=' * 65)
print('Composite task 1: Tool Use + Reasoning')
print('Question: search AI trends + current time + calculate 3^8')
print('=' * 65)
r1 = ask_agent(
    'Search for AI trends, tell me the current time, and then calculate 3 to the power of 8.',
    thread_id='demo'
)
print(f'\nAnswer: {r1[:300]}')

# ── Composite task 2: Saving to Memory ───────────────────────────────────
print('\n' + '=' * 65)
print('Composite task 2: Saving user information (Memory)')
print('=' * 65)
r2 = ask_agent(
    "My name is Junhyuk Lee, and I'm an ML engineer learning LangGraph.",
    thread_id='demo'
)
print(f'\nAnswer: {r2[:150]}')

# ── Composite task 3: Using Memory ───────────────────────────────────────
print('\n' + '=' * 65)
print('Composite task 3: Using memory of the previous conversation')
print('=' * 65)
r3 = ask_agent(
    'What did I say my name was and what do I do?',
    thread_id='demo'
)
print(f'\nAnswer: {r3[:150]}')

[에이전트 모델] qwen/qwen3-next-80b-a3b-instruct (기본 llm=llama-3.1-8b-instruct 대신 사용)
복합 태스크 1: Tool Use + Reasoning
질문: AI 트렌드 검색 + 현재 시각 + 3^8 계산


  [도구] search_web({'query': '최신 AI 트렌드 2024'})
  [도구] get_current_time({'dummy': ''})
  [도구] calculator({'expression': '3 ** 8'})
  [결과] '최신 AI 트렌드 2024' 웹 검색 결과 상위 5건:
1. [AI] 데이터분석, 인공지능 트렌드2024, LLM의 비상, 검색증강생성모델..
  [결과] 2026-07-25 15:58:58
  [결과] 3 ** 8 = 6561

답변: 2024년 최신 AI 트렌드는 다음과 같습니다:

1. **LLM(대형 언어 모델)의 비상**: 검색증강생성(RAG) 모델이 주목받으며, 정확한 정보 제공을 위한 실용적 AI 적용이 확대되고 있습니다.
2. **오픈소스 AI의 약진**: 대기업의 폐쇄적 플랫폼 대신, 오픈소스 모델이 개발자 사이에서 인기를 끌고 있습니다.
3. **멀티모달 AI**: 텍스트, 이미지, 음성 등을 통합해 이해하고 생성하는 AI 기술이 발전 중입니다.
4. **AI 에이전트**: 자율적으로 작업을 계획하고 실행하는 AI 에이전트가 마케팅, 고객 서비스 

복합 태스크 2: 사용자 정보 저장 (Memory)


  [도구] search_web({'query': '최신 AI 트렌드 2024'})
  [도구] get_current_time({'dummy': ''})
  [도구] calculator({'expression': '3 ** 8'})
  [결과] '최신 AI 트렌드 2024' 웹 검색 결과 상위 5건:
1. [AI] 데이터분석, 인공지능 트렌드2024, LLM의 비상, 검색증강생성모델..
  [결과] 2026-07-25 15:58:58
  [결과] 3 ** 8 = 6561

답변: 안녕하세요, 이준혁님! 😊  
LangGraph를 배우는 ML 엔지니어라고 하셨군요, 멋진 선택입니다! LangGraph는 LLM 기반의 복잡한 작업 흐름을 **그래프 기반으로 구성**할 수 있어, 에이전트 시스템, 다단계 추론, 상태 기반 워크플로우 구축에 매우 강력

복합 태스크 3: 이전 대화 기억 활용


  [도구] search_web({'query': '최신 AI 트렌드 2024'})
  [도구] get_current_time({'dummy': ''})
  [도구] calculator({'expression': '3 ** 8'})
  [결과] '최신 AI 트렌드 2024' 웹 검색 결과 상위 5건:
1. [AI] 데이터분석, 인공지능 트렌드2024, LLM의 비상, 검색증강생성모델..
  [결과] 2026-07-25 15:58:58
  [결과] 3 ** 8 = 6561

답변: 이전에 말씀하신 바에 따르면,  
- **이름**: **이준혁**님  
- **직업/역할**: **ML 엔지니어**이며, **LangGraph**를 배우고 계십니다.

맞습니까? 😊  
그렇다면, LangGraph 학습에 도움이 될 만한 예제나 개념 설명이 필요하신가요


---
## Summary

### Learning Summary

| Capability | Key API | Takeaway |
|------|----------|---------|
| **Tool Use** | `@tool`, `bind_tools()`, `ToolMessage` | The LLM selects a tool → executes it → interprets the result |
| **Memory** | `MemorySaver`, `thread_id` | Independent memory per thread, persists across sessions |
| **Planning** | `bind_tools([PydanticSchema])` + `PydanticToolsParser` | Structured plan → dependency-based execution |
| **Reasoning** | CoT prompt, `create_agent` | Higher accuracy through step-by-step reasoning |

### References
- [LangGraph official documentation](https://langchain-ai.github.io/langgraph/)
- [LangChain Tools guide](https://python.langchain.com/docs/how_to/tool_calling/)
- [ReAct paper](https://arxiv.org/abs/2210.03629)
- [Chain of Thought paper](https://arxiv.org/abs/2201.11903)
